# 模型基准测试代码（不保存运行结果文件）

本 notebook 只包含可执行测试代码，不落地任何运行结果文件。


In [ ]:
from pathlib import Path
import sys

# 兼容 VSCode/Jupyter 不同工作目录：向上查找仓库根目录（以 data/ 和 models/ 为标记）
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = None
for c in candidates:
    if (c / 'data').exists() and (c / 'models').exists():
        repo_root = c
        break
if repo_root is None:
    raise RuntimeError('Could not locate repository root containing data/ and models/.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.run_model_benchmark import run_all_benchmarks, build_comparison_table, plot_training_curves

results = run_all_benchmarks(repo_root)
comparison_rows = build_comparison_table(results)

# 基础断言（测试代码）
assert 'kbtl_binary' in results
assert 'kbtl_multiclass' in results
assert 'mjda' in results
assert 'bda_gnat_piper' in results
assert len(comparison_rows) == 4
assert all('status' in row for row in comparison_rows)
print('Benchmark test code executed successfully.')

print('Curve lengths:', {
    'kbtl_binary': len(results['kbtl_binary']['curve']['x']),
    'kbtl_multiclass': len(results['kbtl_multiclass']['curve']['x']),
    'mjda': len(results['mjda']['curve']['x']),
    'bda': len(results['bda_gnat_piper']['curve']['x']),
})


In [ ]:
fig, axes = plot_training_curves(results)
fig


In [ ]:
for row in comparison_rows:
    print(row['metric'], row['status'], f"Python={row['python']:.4f}", f"Ref={row['matlab_ref']:.4f}")
